# browsergraph — one graph, every browser (and none at all)

**Repo:** https://github.com/aidonerightcorp/browsergraph

Write a browser-automation graph **once**. Run it on Playwright, Patchright,
Selenium, undetected-chromedriver, Camoufox — or with **no browser at all**.

This notebook installs straight from GitHub and runs **entirely inside Kaggle**:
no browser, no API keys, no network beyond the install. Everything you see is
the real library, not a simulation.


## Install from the public repo


In [ ]:
%pip install -q git+https://github.com/aidonerightcorp/browsergraph.git

import browsergraph
print('browsergraph', browsergraph.__version__)


## 1. What can this machine actually run?

Browser automation fails for environmental reasons far more often than logical
ones. `doctor` reports what is present and **prints the command that fixes**
anything missing. On Kaggle most engines are absent — that is the point.


In [ ]:
from browsergraph.doctor import run_all
print(run_all().text())


## 2. A graph is nodes over a dimension space

`Spec` is one point: engine x binary x transport x display x stealth x
preprocessing x vision x capture. Nodes never touch an engine directly — they
talk to a 12-method `BrowserPort`. That seam is what makes one graph portable.


In [ ]:
from browsergraph import Graph, Spec, Engine, run
from browsergraph.nodes.actions import Navigate, WaitFor, Click, Extract
from browsergraph.drivers.mock import MockBrowser

PAGES = {'https://acme.example': {'h1': 'Acme Roofing', '#login': 'Log in'}}

graph = (Graph('demo')
         .add(Navigate('https://acme.example'))
         .add(WaitFor('#login'))
         .add(Extract('h1', into='heading')))

res = run(graph, Spec(engine=Engine.MOCK), MockBrowser(pages=PAGES))
print(res.summary())
print('heading:', res.context.data['heading'])


## 3. No browser needed: TLS impersonation

Most pages are server-rendered and need no browser. The catch is that anti-bot
vendors fingerprint the **TLS handshake before any JavaScript runs** — so a
stock Python client is identifiable however good its User-Agent is.

`Engine.HTTP` uses `curl-cffi` to present a real browser's handshake.

Measured on the author's machine against pypi.org:

| engine | time | result |
|---|---|---|
| `HTTP` | **0.35s** | `playwright 1.62.0` |
| `PLAYWRIGHT` | 2.72s | `playwright 1.62.0` |

**7.8x faster, identical output.** And it *refuses* what it cannot do rather
than pretending — a driver that silently no-ops a click surfaces later as
missing data with no explanation.


In [ ]:
from browsergraph.dimensions import Vision, validate
spec = Spec(engine=Engine.HTTP)
print('http spec valid :', validate(spec) == [])
print('http + vision   :', validate(Spec(engine=Engine.HTTP, vision=Vision.ALWAYS)))


## 4. The linter catches the failure that matters most

**BG003 — a graph that changes remote state but never verifies the outcome.**

This rule exists because of a real incident: **551 emails reported "sent"
successfully and produced zero posts.** Every layer said success. Nothing
checked the destination.


In [ ]:
from browsergraph.lint import lint, report

risky = (Graph('risky')
         .add(Navigate('https://acme.example'))
         .add(WaitFor('#login'))
         .add(Click('#login')))       # mutates, never verifies

print(report(lint(risky)))


## 5. Token reduction: 8 strategies, then focus

Raw HTML is mostly framework noise. Preprocessing trades structure against
size; `focus` then keeps only the chunks that answer the question — **plus
their neighbours**, because the chunk matching "contact" is rarely the one
holding the phone number.


In [ ]:
from browsergraph.preprocess import Preprocess, compare, reduce, backends
from browsergraph.focus import focus

HTML = ('<!doctype html><html><head><title>Acme</title>'
        '<style>.a{color:red}</style><script>var big="' + 'z'*4000 + '";</script></head>'
        '<body><nav><a href="/">Home</a></nav><main><h1>Contact Acme</h1>'
        '<p>' + 'Filler about the company. '*80 + '</p>'
        '<h2>Sales</h2><p>Email sales@acme.example or call (303) 555-0142.</p>'
        '<button id=send data-testid=send>Send</button></main>'
        '<footer>(c) Acme</footer></body></html>')

for r in sorted(compare(HTML), key=lambda r: r.chars):
    print(f'{r.strategy.value:<14} {r.chars:>7} chars   saved {r.saved_pct:5.1f}%')

f = focus(reduce(HTML, Preprocess.MARKDOWN).content, 'sales email', budget=500)
print(f'\nfocused to {f.chars} chars (saved {f.saved_pct:.0f}%)')
print('answer kept:', 'sales@acme.example' in f.content)
print('optional backends:', backends())


## 6. Deterministic extraction — conservative on purpose

No model involved. **A false positive silently poisons a dataset; a miss is a
visible empty field.** So dates, repeated digits and asset filenames are all
rejected rather than guessed at.


In [ ]:
from browsergraph.extract.patterns import extract_contacts
from browsergraph.extract.content import text_of

good = extract_contacts(text_of(HTML), [])
print('emails:', good.emails)
print('phones:', [p.raw for p in good.phones])

noise = extract_contacts('Order 2026-03-04, SKU 000000000, logo@2x.png', [])
print('\nnoise -> emails', noise.emails, '| phones', noise.phones)


## 7. NAICS classification with honest confidence

A weak match reports `usable=False` rather than emitting a plausible-looking
code. Classifying *every* input is the failure this avoids.


In [ ]:
from browsergraph.classify.naics import classify

for label, text in [
    ('roofing firm', 'We are a general contractor specialising in roofing and gutters.'),
    ('restaurant',   'Our restaurant menu, dining, reservations and catering.'),
    ('law firm',     'Our law firm attorneys provide legal advisory services.'),
    ('empty page',   'Welcome to our website. Hello.'),
]:
    c = classify(text)
    print(f'{label:<14} {c.code or "--":<6} {c.confidence:<7} usable={c.usable}')


## 8. Control flow lives *inside* the graph

Crawling used to sit outside the graph model — which meant healing,
supervision and the linter did not apply to it, and crawling is where most of
the runtime goes. `branch`, `for_each`, `subgraph`, `frontier` and
`retry_until` are nodes, so every guarantee composes.


In [ ]:
from browsergraph.nodes import REGISTRY
print('node kinds:', sorted(REGISTRY))

from browsergraph.nodes.control import Subgraph
inner = Graph('inner').add(Click('#login'))
outer = Graph('outer').add(Navigate('https://acme.example')).add(Subgraph(inner))
print('\nBG003 still fires through a subgraph:',
      'BG003' in {f.code for f in lint(outer)})


## 9. Don't enumerate the space — sample it

Incompatible combinations are rejected **with reasons**. Full enumeration
explodes, so `sample` builds a pairwise covering array: most failures are
two-value interactions, caught at a fraction of the cost.


In [ ]:
from browsergraph.combos import count, rejected
from browsergraph.sample import coverage, sample_specs
from browsergraph.dimensions import Binary, Display, Stealth

total, ok = count()
print(f'{total} combinations -> {ok} runnable, {total-ok} rejected\n')
for desc, why in rejected()[:4]:
    print(f'  {desc[:58]}\n      {why[0]}')

axes = {'engine': list(Engine), 'binary': list(Binary),
        'display': list(Display), 'stealth': list(Stealth)}
specs = sample_specs(axes)
cov, poss = coverage(axes, specs)
print(f'\npairwise: {len(specs)} runs cover {cov}/{poss} value-pairs')


## 10. Self-tuning: learn from similar sites

Outcomes generalise `site -> org -> sector -> platform -> global`, weighted by
specificity. **Evidence is reported honestly**: one success is `p≈0.67, n=1`
after smoothing, never certainty.


In [ ]:
from browsergraph.learn import Features, Knowledge, Outcome, plan
from browsergraph.strategy import ladder

k, winner = Knowledge(), Spec(engine=Engine.MOCK)
f1 = Features.of('https://acme.example', task='contacts')
k.record(f1, winner, True)
print('one win        :', k.estimate(f1, winner.describe()))

for i in range(6):
    k.record(Features.of(f'https://shop{i}.example', sector='44-45',
                         task='contacts'), winner, True)
fresh = Features.of('https://brandnew.example', sector='44-45', task='contacts')
print('unseen site    :', k.estimate(fresh, winner.describe()))
print('\n' + plan(k, f1, ladder(Spec(engine=Engine.MOCK))).explain())


## 11. Utility, not just success

Binary outcomes can rank *engines*, but not preprocessing or budget — those
rarely change whether a run succeeds, they change what it costs. Recording
yield/tokens/seconds makes those axes learnable.

**Unmeasured success still scores 1.0** — absence of measurement is not
mediocrity.


In [ ]:
cheap = Outcome(ok=True, yield_count=8, tokens=1200, seconds=3)
dear  = Outcome(ok=True, yield_count=8, tokens=90_000, seconds=40)
print('same yield, cheap :', round(cheap.utility(), 3))
print('same yield, costly:', round(dear.utility(), 3))
print('unmeasured success:', Outcome(ok=True).utility())
print('failure           :', Outcome(ok=False, tokens=10).utility())


## 12. Guardrails: never stop on ignorance

Early stopping needs **both** low expected success *and* enough evidence to
trust the number. Abandoning an unfamiliar site on its first failure is
exactly when exploration is worth most.


In [ ]:
from browsergraph.learn import Budget, Estimate

b = Budget(min_expected_success=0.5, min_evidence_to_stop=3)
print('thin evidence  ->', repr(b.should_stop_early(Estimate('s', p=0.1, evidence=1.0))))
print('solid evidence ->', b.should_stop_early(Estimate('s', p=0.1, evidence=8.0)))


## 13. A CAPTCHA is not a missing element

Retrying is not a universal remedy. A bot wall must **abort** — retrying into
one is how accounts get banned. Classification reads the page, not just the
error string, because a challenge usually *presents* as a missing selector.


In [ ]:
from browsergraph.errors import classify as classify_error

cases = [('click target not found: #login', ''),
         ('element not found: #x', 'Please complete the CAPTCHA'),
         ('HTTP 429 too many requests', ''),
         ('403 Forbidden', ''),
         ('timeout waiting for #results', '')]
for err, page in cases:
    d = classify_error(err, page)
    print(f'{d.failure.value:<14} -> {d.response.value:<11} terminal={d.terminal}')


## 14. Politeness belongs where the contention is

A per-crawler delay lets ten concurrent tasks make ten requests per second at
one host. The limiter is **per-domain and process-wide**, and honours robots
`Crawl-delay` when it is stricter — never when it is looser.


In [ ]:
from browsergraph.throttle import DomainPolicy, Limiter

class Clock:
    def __init__(self): self.t = 0.0
    def __call__(self): return self.t
    def sleep(self, s): self.t += s

c, lim = Clock(), Limiter(default=DomainPolicy(min_interval=1.0))
waits = []
for _ in range(4):                       # four independent 'crawlers'
    waits.append(lim.acquire('https://one-host.example/p', sleep=c.sleep, clock=c))
    lim.release('https://one-host.example/p')
print('waits per request:', waits)

lim.observe_crawl_delay('slow.example', 10.0)
lim.observe_crawl_delay('slow.example', 0.1)   # must not relax
print('robots delay honoured:', lim.policy_for('slow.example').min_interval)


## 15. Tasks: say *what*, not *how*


In [ ]:
from browsergraph.tasks import catalog
for t in catalog():
    print(f"{t['name']:<13} {t['summary'][:66]}")


## 16. Multi-model routing by capability

Selection is by the model's **reported capability**, not its name. A vision
job answered by a text model returns confident fiction, so an unqualified
model raises rather than being silently substituted.

On the author's Ollama host this resolves to:

```
classify -> deepseek-v4-flash    selector -> kimi-k2.7-code
plan     -> glm-5.2               vision   -> kimi-k2.7-code
```


In [ ]:
from browsergraph.routing import JOBS
for job, spec in JOBS.items():
    print(f"  {job:<10} needs={spec['capability']:<11} prefers={spec['role'] or '-'}")


## 17. Run it for real

Kaggle has no browser, so this is the one cell you run elsewhere.


In [ ]:
# pip install 'browsergraph[playwright] @ git+https://github.com/aidonerightcorp/browsergraph.git'
# playwright install chromium
#
# from browsergraph.tasks import make
# from browsergraph.drivers import build
# spec = Spec(engine=Engine.PLAYWRIGHT)
# print(make('research', url='https://example.com').run(build(spec)).to_dict())
print('see the repo README for live examples')


---

## Read more

- **Repo**: https://github.com/aidonerightcorp/browsergraph
- [`ARCHITECTURE.md`](https://raw.githubusercontent.com/aidonerightcorp/browsergraph/main/ARCHITECTURE.md) — the Protocol-vs-base-class seam
- [`DIMENSIONS.md`](https://raw.githubusercontent.com/aidonerightcorp/browsergraph/main/DIMENSIONS.md) — axes worth adding, and why verification matters most
- [`ISOLATION.md`](https://raw.githubusercontent.com/aidonerightcorp/browsergraph/main/ISOLATION.md) — conflicting engines in separate virtualenvs
- [`PLUGINS.md`](https://raw.githubusercontent.com/aidonerightcorp/browsergraph/main/PLUGINS.md) — the open plugin format

**428 tests**, stdlib-only core, MIT licensed.
